# Regresión — optimizar, guardar y usar el modelo

En el cuaderno anterior probamos varios algoritmos. Aquí intentamos exprimir más rendimiento del mejor de ellos, mediante dos vías:

1. **Ajuste de hiperparámetros** (búsqueda en cuadrícula).
2. **Preprocesamiento de los datos** (escalado y codificación).

Y al final guardamos el modelo entrenado para poder usarlo después sin reentrenar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

# Cargar y dividir igual que en los cuadernos anteriores
bike_data = pd.read_csv('../datasets/daily-bike-share.csv')
bike_data['day'] = pd.DatetimeIndex(bike_data['dteday']).day

X, y = bike_data[['season', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit',
                  'temp', 'atemp', 'hum', 'windspeed']].values, bike_data['rentals'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

print(f'Entrenamiento: {X_train.shape[0]} filas · Prueba: {X_test.shape[0]} filas')

In [ ]:
resultados = []

def evaluar(model, nombre, mostrar_grafico=True):
    """Evalúa un modelo YA entrenado y registra sus métricas."""
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    print(f'--- {nombre} ---')
    print(f'MAE : {mae:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'R2  : {r2:.4f}\n')

    if mostrar_grafico:
        plt.scatter(y_test, predictions)
        plt.xlabel('Labels reales')
        plt.ylabel('Labels predichos')
        plt.title(nombre)
        z = np.polyfit(y_test, predictions, 1)
        p = np.poly1d(z)
        plt.plot(y_test, p(y_test), color='magenta')
        plt.show()

    resultados.append({'modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

## Punto de partida: Gradient Boosting con valores por defecto

In [ ]:
modelo_base = GradientBoostingRegressor(random_state=0).fit(X_train, y_train)
evaluar(modelo_base, 'GradientBoosting (por defecto)')

## Optimizar hiperparámetros

Fíjate en la definición del estimador: incluye un montón de parámetros que controlan cómo se entrena el modelo.

Cuidado con el vocabulario: en machine learning, *parámetros* son los valores que se **determinan a partir de los datos**. Los valores que tú especificas para afectar el comportamiento del algoritmo se llaman **hiperparámetros**.

¿Cómo saber qué valores usar? Sin un conocimiento profundo del algoritmo, hay que **experimentar**. Scikit-learn ofrece una forma de hacerlo automáticamente: probar múltiples combinaciones y quedarse con la que dé mejor resultado según una métrica.

Vamos a usar **búsqueda en cuadrícula** (*grid search*) sobre dos hiperparámetros de `GradientBoostingRegressor`:

- **`learning_rate`** (velocidad de aprendizaje): cuánto se ajusta el modelo en cada ciclo. Si es muy alta, los ajustes pueden ser tan grandes que el modelo nunca termine de afinarse.
- **`n_estimators`**: cuántos árboles construir.

Con 3 valores de cada uno son 9 combinaciones, y con `cv=3` cada una se prueba 3 veces → 27 entrenamientos. Puede tardar un poco.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

alg = GradientBoostingRegressor(random_state=0)

# Combinaciones a probar
params = {
    'learning_rate': [0.1, 0.5, 1.0],
    'n_estimators': [50, 100, 150]
}

# Buscar la mejor combinación optimizando R2
score = make_scorer(r2_score)
gridsearch = GridSearchCV(alg, params, scoring=score, cv=3, return_train_score=True)
gridsearch.fit(X_train, y_train)

print('Mejor combinación:', gridsearch.best_params_, '\n')

modelo_ajustado = gridsearch.best_estimator_
evaluar(modelo_ajustado, 'GradientBoosting (hiperparámetros ajustados)')

> **Nota**: el ajuste de hiperparámetros no siempre produce una mejora espectacular. Aquí el modelo con valores por defecto ya era bastante bueno, así que la ganancia puede ser pequeña. La técnica sigue siendo importante — en otros escenarios la diferencia sí es grande.

## Preprocesar los datos

Hasta ahora entrenamos con los datos tal cual venían del archivo. En la práctica es común transformarlos antes, para que al algoritmo le resulte más fácil ajustarse.

### Escalar las features numéricas

Normalizar las numéricas para que estén en la misma escala evita que las de valores grandes produzcan coeficientes que afecten desproporcionadamente a las predicciones. Por ejemplo:

| A | B | C |
|---|---|---|
| 3 | 480 | 65 |

Si A va de 0 a 10, B de 0 a 1000 y C de 0 a 100, escalarlas daría:

| A | B | C |
|---|---|---|
| 0.3 | 0.48 | 0.65 |

Hay varias formas: calcular mínimo y máximo de cada columna y asignar un valor proporcional entre 0 y 1 (`MinMaxScaler`), o usar la media y la desviación estándar para mantener la misma *dispersión* en otra escala (`StandardScaler`).

### Codificar las variables categóricas

Los modelos trabajan mejor con números que con texto. Dos técnicas:

**Codificación ordinal** — un entero único por categoría:

| Talla | → |
|---|---|
| S | 0 |
| M | 1 |
| L | 2 |

**One-hot encoding** — una columna binaria por cada valor posible:

| Talla_S | Talla_M | Talla_L |
|---|---|---|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |

La diferencia importa: la ordinal introduce un **orden** artificial (sugiere que L > M > S numéricamente), lo cual tiene sentido para tallas pero no para colores o estaciones del año.

### Pipelines

Para aplicar estas transformaciones usamos **pipelines** de scikit-learn: definen una secuencia de pasos de preprocesamiento que termina en un algoritmo. Se ajusta el pipeline entero a los datos, de modo que el modelo encapsula tanto el preprocesamiento como la regresión.

Esto importa porque al predecir con datos nuevos hay que aplicar **exactamente las mismas transformaciones** (basadas en las mismas distribuciones y codificaciones del entrenamiento). El pipeline lo hace automáticamente.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Preprocesamiento de columnas numéricas (escalarlas)
# Los índices corresponden a las columnas de X: 6=temp, 7=atemp, 8=hum, 9=windspeed
numeric_features = [6, 7, 8, 9]
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Preprocesamiento de columnas categóricas (codificarlas)
# 0=season, 1=mnth, 2=holiday, 3=weekday, 4=workingday, 5=weathersit
categorical_features = [0, 1, 2, 3, 4, 5]
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinar ambos preprocesamientos
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Pipeline completo: preprocesamiento + algoritmo
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(random_state=0))
])

modelo_pipeline = pipeline.fit(X_train, y_train)
print(modelo_pipeline)

El modelo ya está entrenado, preprocesamiento incluido. Veamos cómo rinde.

In [ ]:
evaluar(modelo_pipeline, 'GradientBoosting + preprocesamiento')

El pipeline se compone de las transformaciones más el algoritmo. Para probar otro algoritmo basta cambiar ese último paso:

In [ ]:
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=0))
])

modelo_rf = pipeline_rf.fit(X_train, y_train)
evaluar(modelo_rf, 'RandomForest + preprocesamiento')

## Comparativa final

In [ ]:
df_resultados = pd.DataFrame(resultados).sort_values('R2', ascending=False).reset_index(drop=True)
df_resultados.round(4)

### Dos resultados que conviene entender

Si tus números se parecen a los míos, van a pasar dos cosas que **contradicen la intuición**:

**1. La búsqueda en cuadrícula no mejoró nada.** Devolvió `learning_rate=0.1, n_estimators=100`... que son exactamente los valores por defecto. No es un fallo: significa que scikit-learn ya trae valores razonables y que, en esta cuadrícula concreta, no había nada mejor. Ampliar la cuadrícula (más valores, más hiperparámetros) podría encontrar algo, a costa de más tiempo de cómputo.

**2. El preprocesamiento empeoró ligeramente el modelo.** Esto sí tiene una explicación de fondo:

> **Los modelos basados en árboles no se benefician del escalado.** Un árbol decide con preguntas del tipo "¿es esta feature menor que X?". Ese corte funciona igual con la feature en su escala original que reescalada — el orden de los valores no cambia. En cambio, los modelos **lineales** y los basados en **distancias** (K-Means, regresión logística) sí lo necesitan, porque ahí la magnitud de cada feature pesa directamente en el cálculo.

O sea: escalar no es un paso que se aplique siempre "por si acaso". Depende de qué tipo de algoritmo vayas a usar.

## Usar el modelo entrenado

Elegimos el mejor modelo y lo **guardamos** en disco. Así se puede cargar después sin reentrenar — que es exactamente lo que hace una aplicación real al hacer inferencia.

In [ ]:
import joblib

# Elegir el modelo con mejor R2 de los que entrenamos
modelos_entrenados = {
    'GradientBoosting (por defecto)': modelo_base,
    'GradientBoosting (hiperparámetros ajustados)': modelo_ajustado,
    'GradientBoosting + preprocesamiento': modelo_pipeline,
    'RandomForest + preprocesamiento': modelo_rf,
}

mejor_nombre = df_resultados.iloc[0]['modelo']
mejor_modelo = modelos_entrenados[mejor_nombre]
print(f'Mejor modelo: {mejor_nombre} (R2 = {df_resultados.iloc[0]["R2"]:.4f})')

# Guardarlo como archivo pickle
filename = '../entregables/bike-share-model.pkl'
joblib.dump(mejor_modelo, filename)
print(f'Guardado en: {filename}')

Ahora podemos cargarlo cuando haga falta y usarlo para predecir con datos nuevos. Eso es la **inferencia**.

In [ ]:
# Cargar el modelo desde el archivo
loaded_model = joblib.load(filename)

# Una observación nueva (por ejemplo, la previsión de mañana)
# [season, mnth, holiday, weekday, workingday, weathersit, temp, atemp, hum, windspeed]
X_new = np.array([[1, 1, 0, 3, 1, 1, 0.226957, 0.22927, 0.436957, 0.1869]]).astype('float64')
print('Observación nueva:', list(X_new[0]))

result = loaded_model.predict(X_new)
print(f'Predicción: {np.round(result[0]):.0f} alquileres')

El método `predict` acepta un array de observaciones, así que también sirve para predecir en lote. Por ejemplo, con la previsión meteorológica de los próximos cinco días:

In [ ]:
# Features basadas en una previsión a cinco días
X_new = np.array([
    [0, 1, 1, 0, 0, 1, 0.344167, 0.363625, 0.805833, 0.160446],
    [0, 1, 0, 1, 0, 1, 0.363478, 0.353739, 0.696087, 0.248539],
    [0, 1, 0, 2, 0, 1, 0.196364, 0.189405, 0.437273, 0.248309],
    [0, 1, 0, 3, 0, 1, 0.2,      0.212122, 0.590435, 0.160296],
    [0, 1, 0, 4, 0, 1, 0.226957, 0.22927,  0.436957, 0.1869]
])

results = loaded_model.predict(X_new)
print('Predicción de alquileres a 5 días:')
for i, prediction in enumerate(results, 1):
    print(f'  Día {i}: {np.round(prediction):.0f} alquileres')

## Resumen

En este cuaderno:

1. Partimos del mejor algoritmo del cuaderno anterior.
2. Ajustamos sus **hiperparámetros** con búsqueda en cuadrícula.
3. Añadimos **preprocesamiento** (escalado + one-hot) dentro de un pipeline.
4. Comparamos todas las variantes con MAE, RMSE y R².
5. **Guardamos** el mejor modelo y lo usamos para predecir datos nuevos.

Con esto se cierra el ciclo completo de un proyecto de regresión: explorar → entrenar baseline → probar algoritmos → optimizar → guardar → inferir.